# Занятие 9, пара 1. Тренды ИИ в цифрах


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Проверяем громкое утверждение числами

«ИИ растёт невероятными темпами». Звучит убедительно, но что именно растёт: число моделей,
вычисления на обучение, число пользователей или качество ответов? Сегодня возьмём один измеримый показатель
и проверим, насколько аккуратно его можно описать.

Продолжаем 08: вопрос → таблица → график → ограничение вывода. Нужны Pandas и Matplotlib.
Данные Epoch AI уже лежат локально. После пары сможете читать логарифмическую шкалу,
сравнивать оценки вычислений и проверять знаменатель в сообщениях об «открытых моделях».
Итог — график, таблица по годам и три коротких вывода с оговорками.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Какие именно данные перед нами

**Epoch AI** собирает сведения о заметных моделях: публикации, организации, оценки вычислений и доступность.
Критерии отбора и методика — [на странице источника](https://epoch.ai/data/ai-models)
и в [документации](https://epoch.ai/data/ai-models-documentation).
Это выборка заметных моделей, а не список всех студенческих, научных и коммерческих экспериментов.

В комплекте сохранены девять полей исходной таблицы, все её строки и пропуски.
Лицензия CC BY 4.0, автор Epoch AI; происхождение и контрольные суммы —
`course_support/data/trends/SOURCE.md` и `snapshot.json`.
Снимок не обновляется сам: одинаковый код должен читать одинаковые данные.

**Training compute (FLOP)** — оценка числа операций с плавающей точкой при обучении.
FLOP/s — операции в секунду, то есть скорость вычислителя. Это разные единицы.
Больше вычислений не означает автоматически лучшее качество: важны данные, алгоритм, задача и способ оценки.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

models = pd.read_csv(DATA_DIR / 'trends' / 'notable_models.csv')
print('Строк в снимке:', len(models))
display(models[['Model', 'Publication date', 'Training compute (FLOP)', 'Confidence']].head())


### 2. Приведём даты и выберем границы вопроса

Исследуем исторический период **2010–2025**. Последний неполный год не смешиваем с полными.
Это фиксированные границы упражнения, а не команда менять их каждый семестр.

`pd.to_datetime` преобразует текст в даты. `errors='coerce'` заменяет нераспознанную дату на `NaT` —
пропущенную дату; после преобразования такие случаи надо посчитать. `.dt.year` извлекает год.
Условия отбора знакомы по предобработке. Отдельный `copy()` оставляет исходную таблицу нетронутой.


In [ ]:
models['date'] = pd.to_datetime(models['Publication date'], errors='coerce')
models['year'] = models['date'].dt.year
print('Не прочитано дат:', models['date'].isna().sum())
in_period = (models['year'] >= 2010) & (models['year'] <= 2025)
period = models.loc[in_period].copy()
period['year'] = period['year'].astype(int)
print('Строк в выбранном периоде:', len(period))


Нам нужны положительные оценки вычислений. `pd.to_numeric(..., errors='coerce')` делает для чисел то,
что `to_datetime` сделал для дат. Пропуск нельзя заменить нулём: «мы не знаем затраты» и «затрат не было»
не одно и то же. Посчитаем, какая доля записей вообще участвует в численном анализе.

`to_frame(имя)` превратит результат подсчёта Series в таблицу с именованным столбцом. При добавлении второго столбца Pandas сопоставляет годы по индексу. Ноль fillna ниже относится только к числу известных оценок, не к неизвестным затратам модели.


In [ ]:
period['compute'] = pd.to_numeric(period['Training compute (FLOP)'], errors='coerce')
measured = period.loc[period['compute'] > 0].copy()
coverage = period.groupby('year').size().to_frame('all_records')
coverage['with_compute'] = measured.groupby('year').size()
coverage['with_compute'] = coverage['with_compute'].fillna(0).astype(int)
coverage['known_share'] = coverage['with_compute'] / coverage['all_records']
display(coverage)


`to_frame('all_records')` превратил Series количеств в таблицу с одним именованным столбцом.
При присваивании второго столбца Pandas сопоставил **годы по индексу**, а не строки по порядку.
Ноль после fillna относится к **числу известных оценок в году**, а не к вычислениям неизвестной модели.
Это принципиальная разница. Прежде чем объяснять рост, проверяем, не изменилось ли покрытие.


### 3. Когда линейная шкала мешает увидеть данные

Оценки различаются на много порядков. На обычной оси маленькие значения прижмутся к нулю.
`ax.set_yscale('log')` меняет только отображение, не исходные числа.
На логарифмической шкале расстояние от 10 до 100 такое же, как от 100 до 1000:
каждый раз увеличение в 10 раз. Ноль и отрицательные числа на такой шкале не отображаются.
Сначала проверим это на четырёх вымышленных величинах.


In [ ]:
toy_values = [10, 100, 1000, 10000]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax in axes:
    ax.scatter([1, 2, 3, 4], toy_values)
    ax.set_xlabel('Номер примера')
    ax.set_ylabel('Величина, условные единицы')
axes[0].set_title('Линейная шкала')
axes[1].set_title('Логарифмическая шкала')
axes[1].set_yscale('log')
plt.show()


Теперь тот же приём на настоящих записях. Одна точка — одна модель с известной положительной оценкой.
Точки не соединяем: соседство строк не означает, что одна модель является продолжением другой.
В поле Confidence источник отмечает уверенность в оценке. Это не доверительный интервал качества модели.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax in axes:
    ax.scatter(measured['year'], measured['compute'], alpha=0.45)
    ax.set_xlabel('Год публикации')
    ax.set_ylabel('Оценка вычислений обучения, FLOP')
axes[0].set_title('Линейная шкала')
axes[1].set_title('Те же данные, логарифмическая шкала')
axes[1].set_yscale('log')
fig.tight_layout()
plt.show()


### ✏️ Ваш ход — Один рекорд изменил заголовок

Вымышленный набор затрат: раньше `[10, 10, 10]`, позже `[10, 10, 1000]`.
Редактор написал «типичная модель стала в 34 раза дороже».
Посчитайте `mean_ratio` — отношение средних и `median_ratio` — отношение медиан через `pd.Series(...).mean()/median()`.
Какой именно вывод поддерживает каждое число? Почему даже слово «дороже» не следует напрямую из FLOP?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.1', mean_ratio, median_ratio)


**Мой вывод:** …


### 4. Годовая сводка не отменяет отдельных точек

Посчитаем число известных оценок, медиану и максимум. `agg` с именами уже использовали в 06 и 08.
Годовые медианы сравнивают разные составы моделей; это не повторное измерение одних и тех же объектов.
Максимум отвечает про наиболее затратную **из записанных** моделей, а не про типичную.


In [ ]:
yearly = measured.groupby('year').agg(
    n=('compute', 'size'),
    median_flop=('compute', 'median'),
    max_flop=('compute', 'max'),
)
display(yearly)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(yearly.index, yearly['median_flop'], marker='o', label='Медиана известных оценок')
ax.plot(yearly.index, yearly['max_flop'], marker='o', label='Максимум известных оценок')
ax.set_yscale('log')
ax.set_xlabel('Год')
ax.set_ylabel('Вычисления обучения, FLOP')
ax.legend()
ax.set_title('Сводка записей Epoch AI, а не гарантированный закон роста')
plt.show()


Для конкретного сравнения выберем два года и поделим величины. Не надо называть это
«во столько раз вырос весь ИИ». Мы сравнили медианы опубликованных оценок в выбранном снимке.
Рядом выведем n: отношение двух чисел не показывает, насколько устойчивы эти числа.


In [ ]:
start_year = 2015
end_year = 2025
selected = yearly.loc[[start_year, end_year]]
ratio = selected.loc[end_year, 'median_flop'] / selected.loc[start_year, 'median_flop']
display(selected)
print('Отношение медиан:', round(ratio, 2))


### 5. Кто делает модели — и что считать одной работой?

В Organization встречаются несколько организаций через запятую. Если посчитать исходные строки как названия,
совместная работа станет отдельной «организацией». Разделим их по шагам.
`.str.split(',')` превращает каждую строку в список, `explode` — каждый элемент списка в отдельную строку,
`.str.strip()` убирает пробелы по краям. Покажем на маленькой таблице, прежде чем трогать основную.


In [ ]:
example = pd.DataFrame({'Model': ['A', 'B'], 'Organization': ['Lab One, Lab Two', 'Lab One']})
example['org'] = example['Organization'].str.split(',')
display(example)
expanded_example = example.explode('org')
expanded_example['org'] = expanded_example['org'].str.strip()
display(expanded_example)


Теперь каждой организации засчитываем участие в совместной работе целиком.
Сумма таких участий может превышать число моделей. Это не ошибка, если именно так определён показатель.
`drop_duplicates(['Model','org'])` не даёт случайному повтору имени удвоить участие в одной записи;
имена моделей здесь используем как ключ только после проверки их уникальности в снимке.


In [ ]:
print('Повторяющихся названий моделей:', period['Model'].duplicated().sum())
assert not period['Model'].duplicated().any()
org_rows = period.dropna(subset=['Organization']).copy()
org_rows['org'] = org_rows['Organization'].str.split(',')
org_rows = org_rows.explode('org')
org_rows['org'] = org_rows['org'].str.strip()
org_rows = org_rows.loc[org_rows['org'] != '']
org_rows = org_rows.drop_duplicates(['Model', 'org'])
participation = org_rows.groupby('org').size().sort_values(ascending=False)
display(participation.head(5))


### ✏️ Ваш ход — Два честных рейтинга

У Lab A две самостоятельные модели, у Lab B одна, ещё одна модель создана совместно A и B.
Запишите словари `full_credit` (каждому участнику по единице) и `fractional_credit`
(единица совместной работы делится поровну). Ключи — `'A'`, `'B'`.
Какой показатель подходит вопросу «сколько раз организация участвовала», а какой сохраняет общее число моделей?
Ни один рейтинг сам по себе не измеряет качество.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.2', full_credit, fractional_credit)


**Мой вывод:** …


### 6. «Открытая модель» — слишком широкое выражение

Проверим конкретное поле **Open model weights?**: доступны ли веса по классификации источника.
Доступные веса не означают автоматически открытые обучающие данные, код или разрешение любого использования.
Ограничения лицензии могут сохраняться. Неизвестную доступность не записываем в «No».

Посчитаем Yes, No и пропуски отдельно. `.eq('Yes')` эквивалентен сравнению с этой строкой;
`.fillna(False)` задаёт, что неизвестный ответ не является подтверждённым Yes.
Для итоговой доли среди известных ответов знаменатель — Yes+No.


In [ ]:
status = period['Open model weights?']
yes_count = int(status.eq('Yes').fillna(False).sum())
no_count = int(status.eq('No').fillna(False).sum())
unknown_count = len(status) - yes_count - no_count
print('Yes:', yes_count, 'No:', no_count, 'неизвестно:', unknown_count)
known_count = yes_count + no_count
known_share = yes_count / known_count if known_count else None
print('Доля Yes среди известных:', known_share)


Если хотим долю среди **всех** записей, неизвестность задаёт диапазон.
Нижняя граница: все неизвестные окажутся No. Верхняя: все окажутся Yes.
Это границы из-за пропусков, **не статистический доверительный интервал**.
Пока модель доступа меняется со временем, снимок показывает классификацию источника на момент выгрузки,
а не обязательно условия в день публикации модели.


### ✏️ Ваш ход — Неизвестное нельзя спрятать в знаменатель

Напишите `share_bounds(yes, no, unknown)`, возвращающую пару (нижняя, верхняя)
для доли Yes среди всех записей. Если записей нет, верните `(None, None)`.
Проверьте на 30 Yes, 20 No, 50 неизвестных. Можно ли честно написать «60% всех моделей имеют доступные веса»?
Аргументы — неотрицательные целые количества.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.3', share_bounds)


**Мой вывод:** …


## Сохраним результат и отделим факты от предположений

В своей копии сформулируйте три предложения: что видно на графике; какое ограничение наиболее существенно;
какие дополнительные данные нужны для более сильного вывода. Это часть практики, отдельных баллов за ответы нет.
Следующая основная ячейка сохраняет сводку независимо от упражнений.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'ai_trends'
output_dir.mkdir(parents=True, exist_ok=True)
yearly.to_csv(output_dir / 'yearly_compute.csv')
coverage.to_csv(output_dir / 'coverage.csv')
print(output_dir)


**Резерв на 10–15 минут:** повторите годовую сводку только для записей Confidence='Confident'.
Сначала посчитайте, сколько наблюдений останется в каждом году. Сравните с основным выводом,
не выбирая фильтр только потому, что его график красивее. Более узкая выборка тоже может быть смещена.

Новая лабораторная сегодня не выдаётся. Навыки проверки чисел пригодятся в выводах вашего проекта и в ML.
Следующая пара: даже если знаем объём вычислений, как оценить весь проект с людьми, данными и поддержкой?
